In [1]:
%load_ext autoreload
%autoreload 2

import os
import re
import pickle
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

from src.config import Configuration
from src.model import load_cascade, CascadeClassifier
from maikol_utils.file_utils import list_dir_files

CONFIG = Configuration()

In [2]:
FOLDERS = {
    '999': os.path.join(CONFIG.MODELS_PATH, 'haar_cascades_computed_best_balanced_999'),
    '99':  os.path.join(CONFIG.MODELS_PATH, 'haar_cascades_computed_best_balanced_99'),
    'pfp_99':       os.path.join(CONFIG.MODELS_PATH, 'haar_cascades_computed_best_balanced_pfp_99'),
    'pfp_999':      os.path.join(CONFIG.MODELS_PATH, 'haar_cascades_computed_best_balanced_pfp_999'),
    'pfp_999_all':      os.path.join(CONFIG.MODELS_PATH, 'haar_cascades_computed_best_balanced_pfp_999_all'),
    
}

RE_STAGE_FPR = re.compile(r'haar_cascade_stage_(\d+)_fpr_([\d.]+)')

def parse_name(filepath):
    bn = os.path.basename(filepath).replace('.xml', '')
    m = RE_STAGE_FPR.match(bn)
    if m:
        return int(m.group(1)), float(m.group(2))
    return None, None

all_models = {}   # label -> [(stage, fpr, filepath), ...]
for label, folder in FOLDERS.items():
    files, _ = list_dir_files(folder)
    models = []
    for f in files:
        stage, fpr = parse_name(f)
        if stage is not None:
            models.append((stage, fpr, f))
    models.sort(key=lambda x: x[0])
    all_models[label] = models
    print(f"{label}: {len(models)} stages (1-{models[-1][0] if models else 0})")

999: 28 stages (1-28)
99: 23 stages (1-23)
pfp_99: 18 stages (1-18)
pfp_999: 21 stages (1-21)
pfp_999_all: 21 stages (1-21)


In [3]:
test_faces, tn = list_dir_files(CONFIG.faces_cv_passed_path)
print(f"Test faces loaded: {tn}")

Test faces loaded: 51228


In [4]:
CACHE_PATH = os.path.join(CONFIG.MODELS_PATH, 'eval_cache.pkl')

if os.path.exists(CACHE_PATH):
    print(f"Loading cached results from {CACHE_PATH}")
    with open(CACHE_PATH, 'rb') as f:
        all_results, flat_results = pickle.load(f)
else:
    max_workers = min(32, (os.cpu_count() or 1) * 2)
    cfg = Configuration()

    all_results = {}   # label -> [(stage, fpr, tpr, filepath), ...]
    flat_results = []  # (label, stage, fpr, tpr, filepath)

    for label, models in all_models.items():
        results = []
        for stage, fpr, filepath in models:
            cascade = load_cascade(filepath)
            cfg.crop_size = max(cascade.height, cascade.width)
            clf = CascadeClassifier(cfg, cascade)

            def _predict(path):
                faces, _ = clf.predict(img_path=path, return_candidate_count=True)
                return 1 if len(faces) > 0 else 0

            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                hits = list(tqdm(ex.map(_predict, test_faces), total=tn,
                                 desc=f"{label} s{stage:02d}", leave=False))
            tpr = sum(hits) / tn
            results.append((stage, fpr, tpr, filepath))
            flat_results.append((label, stage, fpr, tpr, filepath))
            print(f"{label:>14s}  stage {stage:02d}:  FPR={fpr:.2e}  TPR={tpr:.4f}\n\n")
        all_results[label] = results

    print(f"Saving results to {CACHE_PATH}")
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump((all_results, flat_results), f)

Loading cached results from ../models/eval_cache.pkl


In [5]:
colors = pc.qualitative.Plotly[:len(all_results)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('FP per Stage', 'TP per Stage'),
    horizontal_spacing=0.08
)

for (label, results), color in zip(all_results.items(), colors):
    stages = [r[0] for r in results]
    fprs   = [max(r[1], 1e-7) * 100 for r in results]
    tprs   = [r[2] * 100 for r in results]

    fig.add_trace(go.Scatter(
        x=stages, y=fprs, mode='lines+markers',
        name=f'{label} FP', legendgroup=label,
        line=dict(color=color, dash='solid'),
        marker=dict(symbol='circle', size=6)), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=stages, y=tprs, mode='lines+markers',
        name=f'{label} TP', legendgroup=label, showlegend=False,
        line=dict(color=color, dash='dash'),
        marker=dict(symbol='circle-open', size=6)), row=1, col=2)

fig.update_yaxes(type='log', tickformat='.1e', title_text='FP (%)', row=1, col=1)
fig.update_yaxes(title_text='TP (%)', range=[-6, 2], row=1, col=1)
fig.update_yaxes(title_text='TP (%)', range=[94.5, 100.5], row=1, col=2)
fig.update_xaxes(title_text='Stage', row=1, col=1)
fig.update_xaxes(title_text='Stage', row=1, col=2)
fig.update_layout(
    height=900,
    title='FP and TP Progression per Cascade Stage',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.05, x=0.5, xanchor='center'),)
fig.show()

In [6]:
best = max(flat_results, key=lambda x: (x[1], x[3]))  # highest stage, then best TPR
best_label, best_stage, best_fpr, best_tpr, best_path = best

print(f"Best model: {best_label} stage {best_stage}")
print(f"  FPR (from file): {best_fpr:.2e}")
print(f"  TPR (evaluated): {best_tpr:.4f}")
print(f"  Path: {best_path}")

Best model: 999 stage 28
  FPR (from file): 0.00e+00
  TPR (evaluated): 0.9833
  Path: ../models/haar_cascades_computed_best_balanced_999/haar_cascade_stage_28_fpr_0.0000000000.xml
